# 01 CDRMIP Pipeline

In [1]:
%load_ext autoreload
%autoreload 2
    
from dask.distributed import Client
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from cdrmip_extremes import configs, pipeline
from cdrmip_extremes.extremes_pipeline import ExtremesPipeline

In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

In [3]:
config = pipeline.PipelineConfig(
    experiments=[
        pipeline.ExperimentConfig(
            experiment_id='1pctCO2-cdr', table_id='Amon', variable_id='tas',
            anomaly_reference='picontrol',grid_label=None,
            crossing_gwls=[1.5, 2.0, 3.0], overshoot=True,
            gwl_match_period=(320, 340),
            rolling_window=21,
        ),
    ],
    realign_years_overrides={'1pctCO2': (0, 140)}
)

In [4]:
config = pipeline.CDRMIPPipeline(config)

In [5]:
pipe = config.run()

=== Loading 1pctCO2-cdr ===
Searching for files matching: {'experiment_id': '1pctCO2-cdr', 'table_id': 'Amon', 'variable_id': 'tas', 'grid_label': None, 'member_id': None}
Loading ACCESS-ESM1-5 / r1i1p1f1
  0241-01-16 12:00:00 -> 1000-12-16 12:00:00
Loading CESM2 / r1i1p1f1
  0001-01-15 12:00:00 -> 0200-12-15 12:00:00
Loading CNRM-ESM2-1 / r1i1p1f2
  1990-01-16 12:00:00 -> 2189-12-16 12:00:00
Loading CanESM5 / r1i1p2f1
  1991-01-16 12:00:00 -> 2290-12-16 12:00:00
Loading GFDL-ESM4 / r1i1p1f1
  0141-01-16 12:00:00 -> 0340-12-16 12:00:00
Loading MIROC-ES2L / r1i1p1f2
  1989-01-16 12:00:00 -> 2490-12-16 12:00:00
Loading NorESM2-LM / r1i1p1f1
  0141-01-16 12:00:00 -> 0399-12-16 12:00:00
Loading UKESM1-0-LL / r1i1p1f2
  1990-01-16 00:00:00 -> 2639-12-16 00:00:00
Searching for ACCESS-ESM1-5 1pctCO2-cdr r1i1p1f1 parent
Loading ACCESS-ESM1-5 1pctCO2 r1i1p1f1
Concatenating 1pctCO2-cdr -> 1pctCO2 [ACCESS-ESM1-5] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabele

In [6]:
pipe.load_gsat().compute_gsat_rolling().compute_peak_warming()

In [7]:
pipe.compute_crossing_years().compute_gwl_matching().extract_gwl_periods()

In [8]:
pipe.compute_ensemble_summaries()

## Save

In [9]:
pipe.save(
    configs.PROC_DIR,
    'CDRMIP',
    array_outputs={
        'anomalies': (pipe.anomaly, 'tas'),
        'gsat': (pipe.gsat_median, 'gsat'),
        'gsat_rolling': (pipe.gsat_rolling_median, 'gsat_rolling'),
        'crossing_years': (pipe.crossing_years_median, 'crossing_years'),
        'gwl_matching_years': (pipe.gwl_matching_median, 'gwl_matching_years'),
        'gwl_differences': (pipe.gwl_differences_median_multimodel, 'gwl_differences'),
        'gwl_reversibility': (pipe.gwl_reversibility_median_multimodel, 'gwl_reversibility'),
    },
    scalar_outputs={
        'peak_warming': pipe.peak_warming,
    },
    overwrite=False
)

## Extreme Temperature Frequency Analysis

In [10]:
ext = ExtremesPipeline(
    regridded_picontrol=pipe.regridded_picontrol,
    regridded_concat=pipe.regridded_concat,
    crossing_years=pipe.crossing_years,
    gwl_matching=pipe.gwl_matching,
    gwl_window=21
)    

In [11]:
ext.run()

In [12]:
ext.load_exceedances('exceedances')
ext.load_exceedances('exceedances_matching')
ext.compute_multi_model_stat('exceedances')
ext.compute_multi_model_stat('exceedances_matching')

In [13]:
ext.compute_branch_differences(source_attr='exceedances_matching')
ext.compute_multi_model_stat('exceedances_matching')
ext.compute_multi_model_stat('branch_differences')
ext.compute_multi_model_stat('branch_differences_matching')

In [14]:
ext.save(
    configs.PROC_DIR,
    'CDRMIP',
    outputs=[
        'agreement',
        'exceedances_median_multimodel',
        'branch_matching_median_multimodel'
        'gwl_period_extremes',
        'agreement_matching',
        'exceedances_matching_median_multimodel',
        'branch_differences_matching_median_multimodel'
        'gwl_period_extremes_matching',
    ],
    overwrite=False
)